[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/magnetic_bottle/magnetic_bottle.ipynb)

# The Magnetic Bottle

A charged particle in a magnetic field circles around the field's lines while it drifts along them. The magnetic force does no work, so the particle's speed never changes; and where the field changes slowly on the scale of the circles, the circling keeps its magnetic moment, the momentum of the circling squared over the field. Moving into a stronger field, where the field's lines crowd together near a coil, the particle must circle faster, which takes speed from its drift, until the drift stops and reverses: the coil is a magnetic mirror, and two of them make a bottle. Particles aimed too closely along the field never run out of drift before the field stops growing, and escape through the ends: the loss cone.

Events and velocities are modelled as antivectors, directions in spacetime, intrinsic, and the magnetic field as a plane of space, a bivector, extrinsic. The field turns the particle's velocity at the rate `velocity.commutator(field)`, and the Lorentz force is the field paired with the velocity by the regressive product, `velocity & field`, a pairing of the intrinsic with the extrinsic rather than a contraction by a metric. A step in the particle's own time is the rotor of that turn, so the speed is kept exactly however long the step.

In the tensor formulation of electrodynamics the equation of motion reads as $\frac{du^\mu}{d\tau} = \frac{q}{m} F^{\mu}{}_{\nu} u^\nu$, and the adiabatic invariant as $\frac{p_\perp^2}{B}$, which keeps the particle out of the mirror where $\sin^2\alpha < \frac{B_{\min}}{B_{\max}}$ for the angle $\alpha$ between its velocity and the field.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.util.animation import show_animation
from examples.electromagnetism.magnetic_bottle import render

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
Bivector = ga.gatype.bivector()                              # Bivector+
# Events, velocities and displacements are antivectors: directions in spacetime, intrinsic.
Antivector = ga.gatype.antivector()                          # Antivector-

# One unit along the lab's time, and unit directions of space, the bottle's axis last.
time = -mv.t.dual()                                          # [] Antivector-
across, sideways, axis = stack([mv.x, mv.y, mv.z]).dual()    # [] Antivector-, Antivector-, Antivector-

## 1. The coils and their field

Each coil is a ring of straight pieces of wire. By Biot–Savart's law each piece contributes the plane it spans with the line to the event, `piece.commutator(separation)`, over the cube of the distance: the field is built from the wire's geometry as a plane directly, with no right-hand rule to pick a perpendicular. The field is weakest midway between the coils and strongest in their planes, and the ratio of the two on the axis sets the loss cone.

In [ ]:
# The coils' radius, their heights along the axis, the straight pieces each is made of, and the current
# in them, which makes the field at the centre two units.
radius = 1.0
heights = np.array([-2.0, 2.0])
n_pieces = 48
current = 1.78
# Points on the axis between the coils the loss cone is read from; the half widths of the view along the
# axis and across it, and the points across it the field's strength is drawn at.
n_axis = 401
view_half_length = 3.0
view_half_width = 1.2
n_grid = 120

In [ ]:
def coils(radius: float, heights: np.ndarray, n_pieces: int) -> tuple[Antivector, Antivector]:
    """Circular coils of `radius` around the axis at each height, as straight pieces of wire: each
    piece's displacement along the current, and its midpoint `[coils * pieces]`."""
    angles = np.linspace(0, 2 * np.pi, n_pieces + 1)
    rim = (mv.xy * (-angles / 2)).exp() >> (across * radius)                   # [pieces + 1] Antivector-
    wire = rim + axis * heights[:, None]                                      # [coils, pieces + 1] Antivector-
    pieces = (wire[:, 1:] - wire[:, :-1]).reshape(len(heights) * n_pieces)
    return pieces, ((wire[:, 1:] + wire[:, :-1]) / 2).reshape(len(heights) * n_pieces)


def field(events: Antivector, pieces: Antivector, midpoints: Antivector) -> Bivector:
    """The magnetic field of steady current in the wire at each event: by Biot–Savart's law, each piece
    contributes the plane it spans with the line to the event, over the cube of the distance."""
    # The event's place in the lab's space.
    position = events - (mv.t & events) * time                                # [...] Antivector-
    separation = position[..., None] - midpoints                              # [..., pieces] Antivector-
    distance = separation.squared().square_root()                             # [..., pieces] Scalar+
    # The commutator of two directions is the bivector that turns one towards the other.
    planes = pieces.commutator(separation)                                    # [..., pieces] Bivector+
    return current * (planes / (distance * distance.squared())).sum(axis=-1)  # [...] Bivector+


pieces, midpoints = coils(radius, heights, n_pieces)
# The field's strength along the axis between the coils, weakest midway and strongest in the coils' planes.
axis_samples = axis * np.linspace(heights[0], heights[-1], n_axis)           # [axis] Antivector-
strengths = field(axis_samples, pieces, midpoints).scalar_norm_squared().square_root()   # [axis] Scalar+
weakest = strengths[strengths.argmin()]                                       # [] Scalar+
strongest = strengths[strengths.argmax()]                                     # [] Scalar+
# The angle to the axis, in radians, inside which a particle at the centre reaches the coils' planes.
loss_cone = (weakest / strongest).square_root().arcsin()                      # [] Scalar+
# The field's strength in the plane through the axis.
axis_points = axis * np.linspace(-view_half_length, view_half_length, 2 * n_grid)            # [along] Antivector-
points = across * np.linspace(-view_half_width, view_half_width, n_grid)[:, None] + axis_points   # [across, along] Antivector-
strength = field(points, pieces, midpoints).scalar_norm_squared().square_root()   # [across, along] Scalar+

print(f"field on the axis from {weakest.to_array():.3f} to {strongest.to_array():.3f}; "
      f"loss cone {np.degrees(loss_cone.to_array()):.1f} degrees")

## 2. Particles in the bottle

The particles start in the bottle's middle plane at a range of angles to the axis, the steepest furthest out. The shallow ones, inside the loss cone, run out through the coils; the steeper ones bounce between them, the steepest turning back soonest, while circling slowly around the axis as the field's lines bend and spread.

In [ ]:
# The particles' speed as a fraction of the speed of light, their angles to the axis in degrees, and
# their distances from the axis at launch.
speed = 0.3
pitches = np.linspace(15, 80, 10)
offsets = np.linspace(0, 0.6, 10)
# The step in the particles' own time, the steps, the steps between the frames drawn, and the duration
# of a frame.
proper_step = 0.1
n_steps = 1000
n_steps_per_frame = 8
duration_ms = 40

In [ ]:
def step(event: Antivector, velocity: Antivector) -> tuple[Antivector, Antivector]:
    """One step in the particles' own time, of unit charge over mass: half a step along the velocity,
    the velocity turned by the field there, and the other half step."""
    halfway = event + velocity * (proper_step / 2)                            # [...] Antivector-
    # The velocity turns at the rate `velocity.commutator(field)`: the rotor of that turn over the step.
    turn = (field(halfway, pieces, midpoints) * (proper_step / 2)).exp()     # [...] Rotor+
    velocity = turn >> velocity                                               # [...] Antivector-
    return halfway + velocity * (proper_step / 2), velocity


def evolve(event: Antivector, velocity: Antivector) -> Iterator[tuple[Antivector, Antivector]]:
    """The particles' events and velocities at every step, without end."""
    while True:
        yield event, velocity
        event, velocity = step(event, velocity)


# Each particle's direction is the axis tilted sideways by its angle; its velocity is the unit direction
# in spacetime of its speed along that direction.
directions = (mv.zy * (-np.radians(pitches) / 2)).exp() >> axis                # [particles] Antivector-
launch_velocities = (time + speed * directions) / np.sqrt(1 - speed**2)                  # [particles] Antivector-
events, velocities = zip(*islice(evolve(across * offsets, launch_velocities), n_steps))
events, velocities = stack(list(events)), stack(list(velocities))             # [steps, particles] Antivector-, Antivector-
show_animation(render.animate_flights(events, pitches, points, strength, radius, heights, n_steps_per_frame),
               "magnetic_bottle", duration_ms)

## 3. The magnetic moment

The Lorentz force of a particle circling at right angles to the field is its circling momentum times the field, so the moment, the circling momentum squared over twice the field, is the force squared over twice the field cubed. Read with the field at the particle itself, it swings within each turn wherever the field varies across the particle's circle; averaged over a turn it stays put for the trapped particles through all their bounces, while the escaping ones lose it as they leave the field behind.

In [ ]:
def moment(velocity: Antivector, magnetic: Bivector) -> Scalar:
    """The magnetic moment of the particles' circling, per unit charge over mass: the Lorentz force
    squared over twice the field cubed, the momentum of the circling squared over twice the field."""
    force = velocity & magnetic                                               # [...] Vector+
    strength = magnetic.scalar_norm_squared().square_root()                   # [...] Scalar+
    # The force of a magnetic field is square to the lab's time, and squares to its size squared.
    return force.squared() / (2 * strength * strength.squared())              # [...] Scalar+


magnetic = field(events, pieces, midpoints)                                   # [steps, particles] Bivector+
moments = moment(velocities, magnetic)                                        # [steps, particles] Scalar+
# The moments averaged over a turn, the steps the velocity takes to turn once in the central field, as
# differences of their running sums.
n_turn = int(2 * np.pi / (proper_step * weakest.to_array()))
running = moments.cumsum(0)                                                   # [steps, particles] Scalar+
means = (running[n_turn:] - running[:-n_turn]) / n_turn                       # [steps - turn, particles] Scalar+
escaped = (mv.z & events[-1]).abs() > heights[-1]                             # [particles]

print("angle to the axis, degrees:   ", pitches)
print("escaped:                      ", escaped)
print("largest change of the moment: ", (means / means[0] - 1).abs().to_array().max(axis=0))

In [ ]:
# checks
# The Lorentz force is the dual of the field's turn of the velocity, and the field keeps the speed.
np.testing.assert_allclose((velocities.commutator(magnetic).dual() - (velocities & magnetic)).kernel, 0, atol=1e-10)
np.testing.assert_allclose((mv.t & velocities).to_array(), 1 / np.sqrt(1 - speed**2), atol=1e-11)
# The particles inside the loss cone escape through the coils, those outside it stay between them,
# keeping their magnetic moment averaged over a turn.
np.testing.assert_array_equal(escaped, loss_cone > np.radians(pitches))
np.testing.assert_allclose((means / means[0]).to_array()[:, ~escaped], 1, rtol=0.2)